# Experiment 7: One-Dimensional Shallow-Water Wave

**Aim:** Compare first-order upwind and second-order Lax-Wendroff propagation of a small-amplitude wave packet.

**Numerical method:** Linearized shallow-water system advanced by conservative upwind and Lax-Wendroff schemes.

> **Scope:** The case exposes numerical diffusion, dispersion, CFL stability, and mass conservation.

This Google Colab notebook is self-contained. It reconstructs the tested solver inside the temporary Colab runtime, runs it, displays quantitative plots and an inline field contour, and packages every output file for download.

> **Colab workflow:** Choose a run mode, use **Runtime > Run all**, then download the results ZIP from the final cell. `quick` is only a workflow check; use `lab` or an approved `custom` case for submission.

In [ ]:
#@title 1. Prepare the self-contained Colab runtime

from pathlib import Path
import shutil
import subprocess
import sys
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None


def run_solver(script_path, arguments, output_dir, timeout=1800):
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(script_path), *[str(value) for value in arguments]]
    print("Running:", " ".join(command))
    completed = subprocess.run(command, cwd=output_dir, capture_output=True, text=True, timeout=timeout)
    (output_dir / "solver_stdout.log").write_text(completed.stdout, encoding="utf-8")
    (output_dir / "solver_stderr.log").write_text(completed.stderr, encoding="utf-8")
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or f"Solver exited with code {completed.returncode}")
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return output_dir


def read_csv(path):
    data = np.genfromtxt(path, delimiter=",", names=True, dtype=None, encoding="utf-8")
    return np.atleast_1d(data)


def show_summary(path):
    text = Path(path).read_text(encoding="utf-8").strip()
    display(Markdown(f"### Solver summary\n```text\n{text}\n```"))
    return text


def list_outputs(output_dir):
    rows = []
    for path in sorted(Path(output_dir).iterdir()):
        if path.is_file():
            size = path.stat().st_size
            if size < 1024:
                size_text = f"{size} B"
            elif size < 1024**2:
                size_text = f"{size / 1024:.1f} KB"
            else:
                size_text = f"{size / 1024**2:.1f} MB"
            rows.append((path.name, size_text))
    table = "\n".join(f"| `{name}` | {size} |" for name, size in rows)
    display(Markdown(f"### Generated files\n| File | Size |\n|---|---:|\n{table}"))
    return rows


def read_structured_vtk(path):
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    dims_index = next(i for i, line in enumerate(lines) if line.startswith("DIMENSIONS "))
    _, nx, ny, nz = lines[dims_index].split()
    nx, ny, nz = int(nx), int(ny), int(nz)
    if nz != 1:
        raise ValueError("Only 2D structured VTK files are supported by this notebook preview")
    points_index = next(i for i, line in enumerate(lines) if line.startswith("POINTS "))
    npoints = int(lines[points_index].split()[1])
    points = np.array([[float(v) for v in lines[points_index + 1 + k].split()[:3]] for k in range(npoints)])
    x = points[:, 0].reshape(ny, nx)
    y = points[:, 1].reshape(ny, nx)
    data_index = next(i for i, line in enumerate(lines) if line.startswith("POINT_DATA ")) + 1
    scalars = {}
    vectors = {}
    i = data_index
    while i < len(lines):
        if lines[i].startswith("SCALARS "):
            name = lines[i].split()[1]
            i += 2
            values = np.array([float(lines[i + k].split()[0]) for k in range(npoints)])
            scalars[name] = values.reshape(ny, nx)
            i += npoints
        elif lines[i].startswith("VECTORS "):
            name = lines[i].split()[1]
            i += 1
            values = np.array([[float(v) for v in lines[i + k].split()[:3]] for k in range(npoints)])
            vectors[name] = (values[:, 0].reshape(ny, nx), values[:, 1].reshape(ny, nx))
            i += npoints
        else:
            i += 1
    return x, y, scalars, vectors


def plot_vtk_scalar(path, preferred_field=None):
    path = Path(path)
    if not path.is_file():
        print(f"VTK preview skipped: {path.name} was not generated")
        return
    x, y, scalars, _ = read_structured_vtk(path)
    if not scalars:
        print("VTK preview skipped: no scalar field found")
        return
    field = preferred_field if preferred_field in scalars else next(iter(scalars))
    plt.figure(figsize=(8, 4.8))
    image = plt.contourf(x, y, scalars[field], levels=30)
    plt.colorbar(image, label=field.replace("_", " "))
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"{field.replace('_', ' ').title()} - inline field preview")
    plt.axis("equal")
    plt.tight_layout()
    plt.show()


def make_results_zip(output_dir, experiment_number):
    output_dir = Path(output_dir)
    archive = output_dir.parent / f"experiment_{experiment_number:02d}_results.zip"
    if archive.exists():
        archive.unlink()
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as handle:
        for path in sorted(output_dir.rglob("*")):
            if path.is_file():
                handle.write(path, path.relative_to(output_dir))
    print(f"Results archive: {archive} ({archive.stat().st_size / 1024:.1f} KB)")
    return archive


EXPERIMENT = 7
ROOT = Path("/content/cfd_lab_exp07") if IN_COLAB else Path.cwd() / "_colab_runtime_exp07"
CODE_DIR = ROOT / "code"
OUTPUT_DIR = ROOT / "outputs"
CODE_DIR.mkdir(parents=True, exist_ok=True)
(CODE_DIR / "common.py").write_text('from __future__ import annotations\n\nimport csv\nimport math\nfrom pathlib import Path\nfrom typing import Iterable, Mapping\n\nimport numpy as np\n\n\ndef trapezoidal_integral(values: np.ndarray, coordinates: np.ndarray) -> float:\n    y = np.asarray(values, dtype=float).ravel()\n    x = np.asarray(coordinates, dtype=float).ravel()\n    if y.size != x.size or y.size < 2:\n        raise ValueError("trapezoidal integration requires equally sized arrays with at least two points")\n    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))\n\n\ndef write_csv(path: str | Path, header: Iterable[str], rows: Iterable[Iterable[object]]) -> None:\n    with Path(path).open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(list(header))\n        writer.writerows(rows)\n\n\ndef mesh_xy(nx: int, ny: int, lx: float = 1.0, ly: float = 1.0, x0: float = 0.0, y0: float = 0.0) -> tuple[np.ndarray, np.ndarray]:\n    x = np.linspace(x0, x0 + lx, nx)\n    y = np.linspace(y0, y0 + ly, ny)\n    return np.meshgrid(x, y)\n\n\ndef write_structured_grid_vtk(\n    path: str | Path,\n    x: np.ndarray,\n    y: np.ndarray,\n    scalars: Mapping[str, np.ndarray] | None = None,\n    vectors: Mapping[str, tuple[np.ndarray, np.ndarray]] | None = None,\n    title: str = "CFD lab output",\n) -> None:\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y, dtype=float)\n    if x.shape != y.shape or x.ndim != 2:\n        raise ValueError("x and y must be equally shaped 2D arrays")\n    ny, nx = x.shape\n    scalar_map = dict(scalars or {})\n    vector_map = dict(vectors or {})\n    for name, values in scalar_map.items():\n        if np.asarray(values).shape != x.shape:\n            raise ValueError(f"scalar {name} has shape {np.asarray(values).shape}, expected {x.shape}")\n    for name, values in vector_map.items():\n        if len(values) != 2 or np.asarray(values[0]).shape != x.shape or np.asarray(values[1]).shape != x.shape:\n            raise ValueError(f"vector {name} has incompatible shape")\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET STRUCTURED_GRID\\n")\n        handle.write(f"DIMENSIONS {nx} {ny} 1\\n")\n        handle.write(f"POINTS {nx * ny} float\\n")\n        for j in range(ny):\n            for i in range(nx):\n                handle.write(f"{x[j, i]:.12e} {y[j, i]:.12e} 0\\n")\n        handle.write(f"POINT_DATA {nx * ny}\\n")\n        for name, values in scalar_map.items():\n            data = np.asarray(values, dtype=float)\n            handle.write(f"SCALARS {name} float 1\\n")\n            handle.write("LOOKUP_TABLE default\\n")\n            for value in data.ravel(order="C"):\n                handle.write(f"{value:.12e}\\n")\n        for name, values in vector_map.items():\n            vx = np.asarray(values[0], dtype=float)\n            vy = np.asarray(values[1], dtype=float)\n            handle.write(f"VECTORS {name} float\\n")\n            for ax, ay in zip(vx.ravel(order="C"), vy.ravel(order="C")):\n                handle.write(f"{ax:.12e} {ay:.12e} 0\\n")\n\n\ndef write_polyline_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, title: str = "polyline", close: bool = False) -> None:\n    x = np.asarray(x, dtype=float).ravel()\n    y = np.asarray(y, dtype=float).ravel()\n    if x.size != y.size or x.size < 2:\n        raise ValueError("polyline requires equally sized x and y arrays")\n    ids = list(range(x.size))\n    if close:\n        ids.append(0)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {x.size} float\\n")\n        for xi, yi in zip(x, y):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES 1 {len(ids) + 1}\\n")\n        handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_multiline_vtk(path: str | Path, lines: list[tuple[np.ndarray, np.ndarray]], title: str = "lines") -> None:\n    points: list[tuple[float, float]] = []\n    connectivity: list[list[int]] = []\n    for x, y in lines:\n        xa = np.asarray(x, dtype=float).ravel()\n        ya = np.asarray(y, dtype=float).ravel()\n        if xa.size != ya.size or xa.size < 2:\n            raise ValueError("each line needs equally sized x and y arrays")\n        ids = list(range(len(points), len(points) + xa.size))\n        points.extend(zip(xa.tolist(), ya.tolist()))\n        connectivity.append(ids)\n    size = sum(len(ids) + 1 for ids in connectivity)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {len(points)} float\\n")\n        for xi, yi in points:\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES {len(connectivity)} {size}\\n")\n        for ids in connectivity:\n            handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_points_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, scalars: Mapping[str, np.ndarray] | None = None, title: str = "points") -> None:\n    xa = np.asarray(x, dtype=float).ravel()\n    ya = np.asarray(y, dtype=float).ravel()\n    if xa.size != ya.size:\n        raise ValueError("point arrays must have equal size")\n    scalar_map = dict(scalars or {})\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {xa.size} float\\n")\n        for xi, yi in zip(xa, ya):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"VERTICES {xa.size} {2 * xa.size}\\n")\n        for i in range(xa.size):\n            handle.write(f"1 {i}\\n")\n        if scalar_map:\n            handle.write(f"POINT_DATA {xa.size}\\n")\n            for name, values in scalar_map.items():\n                data = np.asarray(values, dtype=float).ravel()\n                if data.size != xa.size:\n                    raise ValueError(f"scalar {name} has wrong size")\n                handle.write(f"SCALARS {name} float 1\\n")\n                handle.write("LOOKUP_TABLE default\\n")\n                for value in data:\n                    handle.write(f"{value:.12e}\\n")\n\n\ndef gradient(field: np.ndarray, dx: float, dy: float) -> tuple[np.ndarray, np.ndarray]:\n    gy, gx = np.gradient(field, dy, dx, edge_order=2)\n    return gx, gy\n\n\ndef laplacian(field: np.ndarray, dx: float, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    out[1:-1, 1:-1] = (\n        (field[1:-1, 2:] - 2.0 * field[1:-1, 1:-1] + field[1:-1, :-2]) / (dx * dx)\n        + (field[2:, 1:-1] - 2.0 * field[1:-1, 1:-1] + field[:-2, 1:-1]) / (dy * dy)\n    )\n    return out\n\n\ndef upwind_derivative_x(field: np.ndarray, velocity: np.ndarray, dx: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[1:-1, :-2]) / dx\n    forward = (field[1:-1, 2:] - field[1:-1, 1:-1]) / dx\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef upwind_derivative_y(field: np.ndarray, velocity: np.ndarray, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[:-2, 1:-1]) / dy\n    forward = (field[2:, 1:-1] - field[1:-1, 1:-1]) / dy\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef poisson_jacobi(phi: np.ndarray, rhs: np.ndarray, dx: float, dy: float, iterations: int, boundary: str = "neumann", mask: np.ndarray | None = None) -> np.ndarray:\n    dx2 = dx * dx\n    dy2 = dy * dy\n    denom = 2.0 * (dx2 + dy2)\n    for _ in range(iterations):\n        old = phi.copy()\n        phi[1:-1, 1:-1] = (\n            (old[1:-1, 2:] + old[1:-1, :-2]) * dy2\n            + (old[2:, 1:-1] + old[:-2, 1:-1]) * dx2\n            - rhs[1:-1, 1:-1] * dx2 * dy2\n        ) / denom\n        if mask is not None:\n            phi[mask > 0.5] = 0.0\n        if boundary == "neumann":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = 0.0\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n        elif boundary == "closed":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = phi[:, -2]\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n            phi -= np.mean(phi)\n        else:\n            raise ValueError("unsupported pressure boundary")\n    return phi\n\n\ndef tdma(a: np.ndarray, b: np.ndarray, c: np.ndarray, d: np.ndarray) -> np.ndarray:\n    a = np.asarray(a, dtype=float).copy()\n    b = np.asarray(b, dtype=float).copy()\n    c = np.asarray(c, dtype=float).copy()\n    d = np.asarray(d, dtype=float).copy()\n    n = d.size\n    if n == 0 or a.size != n or b.size != n or c.size != n:\n        raise ValueError("TDMA arrays must have equal nonzero size")\n    for i in range(1, n):\n        if abs(b[i - 1]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        factor = a[i] / b[i - 1]\n        b[i] -= factor * c[i - 1]\n        d[i] -= factor * d[i - 1]\n    x = np.empty(n, dtype=float)\n    if abs(b[-1]) < 1e-30:\n        raise FloatingPointError("zero TDMA pivot")\n    x[-1] = d[-1] / b[-1]\n    for i in range(n - 2, -1, -1):\n        if abs(b[i]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        x[i] = (d[i] - c[i] * x[i + 1]) / b[i]\n    return x\n\n\ndef relative_l2(numerical: np.ndarray, reference: np.ndarray) -> float:\n    a = np.asarray(numerical, dtype=float)\n    b = np.asarray(reference, dtype=float)\n    denominator = float(np.linalg.norm(b.ravel()))\n    numerator = float(np.linalg.norm((a - b).ravel()))\n    return numerator / max(denominator, 1e-30)\n\n\ndef require_finite(*arrays: np.ndarray) -> None:\n    for array in arrays:\n        if not np.all(np.isfinite(array)):\n            raise FloatingPointError("non-finite numerical field detected")\n\n\ndef positive_float(value: str, name: str) -> float:\n    result = float(value)\n    if not math.isfinite(result) or result <= 0.0:\n        raise ValueError(f"{name} must be positive and finite")\n    return result\n\n\ndef positive_int(value: str, name: str, minimum: int = 1) -> int:\n    result = int(value)\n    if result < minimum:\n        raise ValueError(f"{name} must be at least {minimum}")\n    return result\n', encoding="utf-8")
(CODE_DIR / 'exp07_lake_wave_hyperbolic.py').write_text('from __future__ import annotations\n\nimport csv\nimport math\nimport sys\n\nimport numpy as np\n\nfrom common import relative_l2, trapezoidal_integral, write_structured_grid_vtk\n\n\ndef gaussian(x: np.ndarray, center: float, sigma: float) -> np.ndarray:\n    return np.exp(-0.5 * ((x - center) / sigma) ** 2)\n\n\ndef analytical_eta(x: np.ndarray, time: float, speed: float, amplitude: float, center: float, sigma: float) -> np.ndarray:\n    return 0.5 * amplitude * (gaussian(x, center + speed * time, sigma) + gaussian(x, center - speed * time, sigma))\n\n\ndef upwind(field: np.ndarray, courant: float) -> np.ndarray:\n    if courant >= 0.0:\n        return field - courant * (field - np.roll(field, 1))\n    return field - courant * (np.roll(field, -1) - field)\n\n\ndef lax_wendroff(field: np.ndarray, courant: float) -> np.ndarray:\n    return field - 0.5 * courant * (np.roll(field, -1) - np.roll(field, 1)) + 0.5 * courant * courant * (np.roll(field, -1) - 2.0 * field + np.roll(field, 1))\n\n\ndef reconstruct(wp: np.ndarray, wm: np.ndarray, depth: float, speed: float) -> tuple[np.ndarray, np.ndarray]:\n    velocity = 0.5 * (wp + wm)\n    eta = depth * (wp - wm) / (2.0 * speed)\n    return eta, velocity\n\n\ndef write_visual(path: str, x: np.ndarray, eta: np.ndarray, velocity: np.ndarray, depth: float, scale: float, nz: int) -> None:\n    vertical_fraction = np.linspace(0.0, 1.0, nz)\n    xx = np.repeat(x[None, :], nz, axis=0)\n    bottom = -depth * scale\n    surface = eta * scale\n    yy = bottom + vertical_fraction[:, None] * (surface[None, :] - bottom)\n    eta_field = np.repeat(eta[None, :], nz, axis=0)\n    velocity_field = np.repeat(velocity[None, :], nz, axis=0)\n    fraction = np.repeat(vertical_fraction[:, None], x.size, axis=1)\n    write_structured_grid_vtk(path, xx, yy, {"surface_elevation": eta_field, "vertical_fraction": fraction, "velocity_magnitude": np.abs(velocity_field)}, {"velocity": (velocity_field, np.zeros_like(velocity_field))}, "Still-lake shallow-water visualization")\n\n\ndef write_surface_ribbon(path: str, x: np.ndarray, eta: np.ndarray, scale: float) -> None:\n    xx = np.repeat(x[None, :], 2, axis=0)\n    yy = np.vstack((eta * scale - 0.01 * scale, eta * scale + 0.01 * scale))\n    eta_field = np.repeat(eta[None, :], 2, axis=0)\n    write_structured_grid_vtk(path, xx, yy, {"surface_elevation": eta_field}, title="Lake free-surface ribbon")\n\n\ndef main() -> int:\n    nx = int(sys.argv[1]) if len(sys.argv) > 1 else 401\n    length = float(sys.argv[2]) if len(sys.argv) > 2 else 20.0\n    depth = float(sys.argv[3]) if len(sys.argv) > 3 else 1.0\n    amplitude = float(sys.argv[4]) if len(sys.argv) > 4 else 0.02\n    sigma = float(sys.argv[5]) if len(sys.argv) > 5 else 0.5\n    center = float(sys.argv[6]) if len(sys.argv) > 6 else 10.0\n    end_time = float(sys.argv[7]) if len(sys.argv) > 7 else 2.0\n    cfl = float(sys.argv[8]) if len(sys.argv) > 8 else 0.8\n    output_every = int(sys.argv[9]) if len(sys.argv) > 9 else 25\n    gravity = float(sys.argv[10]) if len(sys.argv) > 10 else 9.81\n    vertical_scale = float(sys.argv[11]) if len(sys.argv) > 11 else 180.0\n    nz_visual = int(sys.argv[12]) if len(sys.argv) > 12 else 61\n    if nx < 41 or length <= 0.0 or depth <= 0.0 or sigma <= 0.0 or end_time < 0.0 or gravity <= 0.0 or output_every < 1 or nz_visual < 3:\n        raise ValueError("usage: python exp07_lake_wave_hyperbolic.py nx L H amplitude sigma x0 tEnd CFL outputEvery gravity verticalScale nzVisual")\n    cfl = min(0.99, max(0.01, cfl))\n    wave_speed = math.sqrt(gravity * depth)\n    x = np.linspace(0.0, length, nx)\n    dx = x[1] - x[0]\n    dt_nominal = cfl * dx / wave_speed\n    eta0 = amplitude * gaussian(x, center, sigma)\n    wp_up = wave_speed * eta0 / depth\n    wm_up = -wave_speed * eta0 / depth\n    wp_lw = wp_up.copy()\n    wm_lw = wm_up.copy()\n    eta_up, u_up = reconstruct(wp_up, wm_up, depth, wave_speed)\n    eta_lw, u_lw = reconstruct(wp_lw, wm_lw, depth, wave_speed)\n    initial_mass = trapezoidal_integral(eta0, x)\n    snapshot_times = [0.0]\n    snapshot_eta = [eta_lw.copy()]\n    history: list[list[float]] = []\n    time = 0.0\n    step = 0\n    while time < end_time - 1e-14:\n        dt = min(dt_nominal, end_time - time)\n        courant = wave_speed * dt / dx\n        wp_up = upwind(wp_up, courant)\n        wm_up = upwind(wm_up, -courant)\n        wp_lw = lax_wendroff(wp_lw, courant)\n        wm_lw = lax_wendroff(wm_lw, -courant)\n        time += dt\n        step += 1\n        eta_up, u_up = reconstruct(wp_up, wm_up, depth, wave_speed)\n        eta_lw, u_lw = reconstruct(wp_lw, wm_lw, depth, wave_speed)\n        reference = analytical_eta(x, time, wave_speed, amplitude, center, sigma)\n        error_up = relative_l2(eta_up, reference)\n        error_lw = relative_l2(eta_lw, reference)\n        mass_up = trapezoidal_integral(eta_up, x)\n        mass_lw = trapezoidal_integral(eta_lw, x)\n        history.append([step, time, dt, courant, error_up, error_lw, mass_up, mass_lw])\n        if step % output_every == 0 or time >= end_time - 1e-14:\n            write_visual(f"lake_water_body_{step:06d}.vtk", x, eta_lw, u_lw, depth, vertical_scale, nz_visual)\n            write_surface_ribbon(f"lake_surface_scaled_{step:06d}.vtk", x, eta_lw, vertical_scale)\n            snapshot_times.append(time)\n            snapshot_eta.append(eta_lw.copy())\n            print(f"step={step} time={time:.6e} upwind_error={error_up:.6e} lax_wendroff_error={error_lw:.6e}")\n    reference = analytical_eta(x, time, wave_speed, amplitude, center, sigma)\n    write_visual("OPEN_THIS_lake_visual_final.vtk", x, eta_lw, u_lw, depth, vertical_scale, nz_visual)\n    write_surface_ribbon("OPEN_THIS_lake_surface_ribbon_final.vtk", x, eta_lw, vertical_scale)\n    write_visual("lake_water_body_final.vtk", x, eta_lw, u_lw, depth, vertical_scale, nz_visual)\n    write_surface_ribbon("lake_surface_scaled_final.vtk", x, eta_lw, vertical_scale)\n    snapshot_array = np.asarray(snapshot_eta)\n    time_array = np.asarray(snapshot_times)\n    xx = np.repeat(x[None, :], time_array.size, axis=0)\n    tt = np.repeat(time_array[:, None], nx, axis=1)\n    write_structured_grid_vtk("lake_space_time.vtk", xx, tt, {"surface_elevation": snapshot_array}, title="Lake wave space-time field")\n    with open("lake_wave_history.csv", "w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["step", "time", "dt", "CFL", "relative_l2_error_upwind", "relative_l2_error_lax_wendroff", "mass_upwind", "mass_lax_wendroff"])\n        writer.writerows(history)\n    with open("lake_wave_profiles.csv", "w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["x", "eta_upwind", "velocity_upwind", "eta_lax_wendroff", "velocity_lax_wendroff", "eta_analytical"])\n        for values in zip(x, eta_up, u_up, eta_lw, u_lw, reference):\n            writer.writerow(values)\n    with open("lake_wave_summary.txt", "w", encoding="utf-8") as handle:\n        handle.write("Linearized one-dimensional shallow-water equations\\n")\n        handle.write(f"nx {nx}\\nlength_m {length:.12e}\\nmean_depth_m {depth:.12e}\\ngravity {gravity:.12e}\\nwave_speed {wave_speed:.12e}\\n")\n        handle.write(f"initial_amplitude {amplitude:.12e}\\nsigma {sigma:.12e}\\ninitial_center {center:.12e}\\nfinal_time {time:.12e}\\ntime_steps {step}\\nCFL {cfl:.12e}\\n")\n        handle.write(f"initial_eta_mass {initial_mass:.12e}\\nfinal_eta_mass_upwind {trapezoidal_integral(eta_up, x):.12e}\\nfinal_eta_mass_lax_wendroff {trapezoidal_integral(eta_lw, x):.12e}\\n")\n        handle.write(f"relative_L2_error_upwind {relative_l2(eta_up, reference):.12e}\\nrelative_L2_error_lax_wendroff {relative_l2(eta_lw, reference):.12e}\\n")\n    print("Wrote OPEN_THIS_lake_visual_final.vtk, OPEN_THIS_lake_surface_ribbon_final.vtk, lake_space_time.vtk, lake_wave_profiles.csv, lake_wave_history.csv, lake_wave_summary.txt")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
SCRIPT = CODE_DIR / 'exp07_lake_wave_hyperbolic.py'
print("Runtime:", "Google Colab" if IN_COLAB else "local notebook test")
print("Working folder:", ROOT)
print("No package installation or external file upload is required.")


## Parameters

Use `quick` to test the workflow, `lab` for the prescribed case, and `custom` only when changing the form values for the assignment.

In [ ]:
#@title 2. Select run mode and parameters
RUN_MODE = "quick" #@param ["quick", "lab", "custom"]

nx = 401 #@param {type:"integer"}
ny_vtk = 20 #@param {type:"integer"}
depth = 1.0 #@param {type:"number"}
amplitude = 0.02 #@param {type:"number"}
x0 = 0.5 #@param {type:"number"}
length = 10.0 #@param {type:"number"}
t_end = 2.0 #@param {type:"number"}
CFL = 0.8 #@param {type:"number"}
output_every = 25 #@param {type:"integer"}
gravity = 9.81 #@param {type:"number"}
n_snapshots = 180 #@param {type:"integer"}
space_time_rows = 61 #@param {type:"integer"}

QUICK_ARGS = [81, 20, 1.0, 0.02, 0.5, 10.0, 0.1, 0.8, 10, 9.81, 30, 15]
LAB_ARGS = [401, 20, 1.0, 0.02, 0.5, 10.0, 2.0, 0.8, 25, 9.81, 180, 61]
CUSTOM_ARGS = [nx, ny_vtk, depth, amplitude, x0, length, t_end, CFL, output_every, gravity, n_snapshots, space_time_rows]
ARGS = QUICK_ARGS if RUN_MODE == "quick" else LAB_ARGS if RUN_MODE == "lab" else CUSTOM_ARGS
print("Run mode:", RUN_MODE)
print("Arguments:", ARGS)


## Run the solver

In [ ]:
#@title 3. Run the numerical solver
OUT = run_solver(SCRIPT, ARGS, OUTPUT_DIR)


## Numerical summary

In [ ]:
#@title 4. Read the solver summary
summary_text = show_summary(OUT / 'lake_wave_summary.txt')


## Quantitative plots

In [ ]:
#@title 5. Plot the primary numerical results
profiles = read_csv(OUT / "lake_wave_profiles.csv")
plt.figure(figsize=(8, 4))
plt.plot(profiles["x"], profiles["eta_upwind"], label="Upwind")
plt.plot(profiles["x"], profiles["eta_lax_wendroff"], label="Lax-Wendroff")
plt.plot(profiles["x"], profiles["eta_analytical"], "--", label="Analytical")
plt.xlabel("x")
plt.ylabel("Surface displacement")
plt.title("Final wave profile")
plt.legend()
plt.grid(True)
plt.show()

history = read_csv(OUT / "lake_wave_history.csv")
plt.figure(figsize=(7, 4))
plt.semilogy(history["time"], history["relative_l2_error_upwind"], label="Upwind")
plt.semilogy(history["time"], history["relative_l2_error_lax_wendroff"], label="Lax-Wendroff")
plt.xlabel("Time")
plt.ylabel("Relative L2 error")
plt.title("Error evolution")
plt.legend()
plt.grid(True)
plt.show()

## Inline field visualization

This preview removes the need for ParaView during the basic laboratory workflow. The VTK file is still included in the downloaded results for optional advanced post-processing.

In [ ]:
#@title 6. Display the final field and generated files
plot_vtk_scalar(OUT / 'lake_space_time.vtk', 'surface_elevation')
files = list_outputs(OUT)


## Download results

In [ ]:
#@title 7. Package and download the results
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
RESULTS_ZIP = make_results_zip(OUT, 7)
if DOWNLOAD_RESULTS:
    if IN_COLAB:
        colab_files.download(str(RESULTS_ZIP))
    else:
        print("Local test mode. Archive retained at:", RESULTS_ZIP)
else:
    print("Set DOWNLOAD_RESULTS to True and run this cell when ready to download.")


## 7. Compulsory assignment

Run CFL = 0.4, 0.8, and 1.1. Compare stability, amplitude loss, phase error, and mass conservation for both schemes.

## Report checklist

- State the governing equations and boundary conditions.
- Report the complete parameter set and grid.
- Include residual or error history.
- Include the required quantitative comparison.
- Include at least one correctly labelled inline field figure; VTK/ParaView figures are optional unless specifically assigned.
- Distinguish observed numerical behaviour from physical interpretation.
- State the limitations of the supplied teaching model.